In [ ]:
!pip install transformers torch pandas sklearn

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
data = pd.read_excel('/content/Qualitative_coding_emotional_needs_TLR_LH (1).xlsx')

# Assume 'Original Post' is the column with the text data and 'Label' is the binary classification target
X = data['question_content/Original Post']
y = data['Emotional support need']  # Ensure this column is binary [0, 1] as described

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
data.head()

In [ ]:
from transformers import RobertaTokenizer

# Initialize the tokenizer
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

# Tokenize the data
train_encodings = tokenizer(X_train.tolist(), truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(X_test.tolist(), truncation=True, padding=True, max_length=512)

In [ ]:
import torch

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Create the dataset
train_dataset = Dataset(train_encodings, y_train.tolist())
test_dataset = Dataset(test_encodings, y_test.tolist())


In [ ]:
pip install accelerate -U

In [ ]:
pip install transformers[torch] -U

In [ ]:
# Ensure all necessary libraries are imported
import numpy as np
import torch
from transformers import RobertaForSequenceClassification, RobertaTokenizer, Trainer, TrainingArguments
from sklearn.metrics import classification_report

# Attempt to import accelerate, handle possible ImportError
try:
    from accelerate import Accelerator
except ImportError:
    print("Accelerate library is not installed. Installing now...")
    !pip install accelerate -U

In [ ]:
# Ensure all necessary libraries are imported
import numpy as np
import torch
from transformers import RobertaForSequenceClassification, RobertaTokenizer, Trainer, TrainingArguments
from sklearn.metrics import classification_report

In [ ]:
# Load the tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)

In [ ]:
pip install transformers[torch]

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./content/results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./content/logs',
    logging_steps=10,
)

In [ ]:
# Ensure all necessary libraries are imported
import numpy as np
import torch
from transformers import RobertaForSequenceClassification, RobertaTokenizer, Trainer, TrainingArguments
from sklearn.metrics import classification_report

# Attempt to import accelerate, handle possible ImportError
try:
    from accelerate import Accelerator
except ImportError:
    print("Accelerate library is not installed. Installing now...")
    !pip install accelerate -U

# Load the tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

# Create an instance of Trainer and proceed with training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()


In [ ]:
from transformers import RobertaForSequenceClassification, Trainer, TrainingArguments

# Load the model
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)

# Training arguments
training_args = TrainingArguments(
    output_dir='./content/results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=8,   # batch size for training
    per_device_eval_batch_size=16,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./content/logs',            # directory for storing logs
    logging_steps=10,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# Train the model
trainer.train()


In [ ]:
from sklearn.metrics import classification_report

# Make predictions
preds_output = trainer.predict(test_dataset)

# Evaluate predictions
preds = np.argmax(preds_output.predictions, axis=-1)
print(classification_report(y_test, preds, target_names=['No Support', 'Support']))